# Embedding Failure-Case Analysis

This stage examines the 15 rank-1 failures identified in the baseline retrieval evaluation and compares their rankings under the two embedding models tested in notebook 06.

The purpose is to determine whether the small aggregate differences between MiniLM and PubMedBERT correspond to meaningful fixes in the known failure cases.

## Controlled comparison

The comparison keeps the indexed chunks and queries unchanged. Only the embedding model changes.

For each of the 15 previously identified failures, we record the rank of the correct PubMed source under each model and identify cases where the ranking changes.

A model that improves aggregate metrics but does not improve the known failure cases would be weaker evidence for switching. Conversely, improvements in the difficult cases would provide qualitative support for a model change.

## Run

From the repository root:

```bash
python src/compare_failure_cases.py
```

The script saves the comparison to `data/embedding_failure_case_comparison.json`.

## What to look for

Focus on three categories:

1. Cases where PubMedBERT moves the correct source from outside the top 5 into the top 5.
2. Cases where it improves a rank-2/3/4 near-miss to rank 1.
3. Cases where PubMedBERT changes a previously good ranking for the worse.

The decision should consider both improvements and regressions rather than counting only the wins.

## Decision point

After running the comparison, use the observed changes to decide whether the embedding model is worth changing. If the changes are mixed or marginal, keep MiniLM and move to the next retrieval improvement rather than optimizing the embedding model indefinitely.

If PubMedBERT clearly fixes several difficult failures with few or no meaningful regressions, it becomes a stronger candidate for the project baseline.

## Results

The failure-case comparison shows a much more informative picture than the aggregate metrics alone.

PubMedBERT improves the correct-source rank to rank 1 in 7 of the 15 previously identified failure cases. These include several of the difficult baseline misses: `18359123` moves from rank 240 to rank 4, `20064872` from rank 7 to rank 1, `21398266` from rank 15 to rank 1, and `23810330` from rank 6 to rank 1. It also improves `14697414`, `12805495`, and `12612531` from rank 2 to rank 1.

There are also meaningful regressions. `11570976` moves from rank 3 to rank 32, `24434052` from rank 2 to rank 41, and `10411439` improves substantially from rank 451 to 159 but still remains outside the top five. Other smaller regressions move correct results from rank 2 to ranks 3–5.

The most important pattern is that PubMedBERT is substantially better on several of the hardest biomedical retrieval cases, while also creating a few serious regressions on ambiguous or broadly phrased questions.

## Decision

The evidence now supports using `neuml/pubmedbert-base-embeddings` as the leading embedding candidate for this biomedical RAG project, despite its slightly worse aggregate MRR in the initial benchmark.

The reason is not simply that it is a biomedical model. The failure-case analysis shows that it can recover several documents that MiniLM ranks very poorly, including four of the five baseline top-five misses that improved into the top five.

However, this should be treated as a candidate decision rather than a final optimization result. The regressions to ranks 32 and 41 show that biomedical embeddings do not solve every retrieval problem. Before locking the model into the production-style pipeline, we should test a retrieval strategy that combines candidate generation with reranking. This can help preserve broad recall while improving the ordering of semantically similar candidates.

The next stage should therefore be a reranking experiment using the same queries and corpus, with PubMedBERT as the candidate-generation embedding model. Keep MiniLM available as the baseline for comparison.